# E2 — Response prediction (top-k lift): "Who is likely to visit/convert?"

**Question.** Can we predict, from anonymized user features alone, who will respond — i.e. P(Y=1 | X)?

**Business decision this informs.** This is the practitioner-default targeting rule: rank users by predicted response and act on the top k% (here k=10). It is deliberately the "wrong-but-useful" question — later (E3+) it is contrasted with uplift targeting, which asks *conditional on treatment* ("who responds *because* of the ad"), not P(Y=1|X).

**Population.** All Criteo uplift v2 users. Models are fit on TRAIN-split *treated* rows only; all selection/evaluation on the VALIDATION split (contains treated + control rows; control rows are never used for fitting — treated-only fitting models response in the ad-enabled environment, since control rows would contaminate calibration with Y(0)).

**Outcome.** Primary: `visit` (validation base rate 4.70%). Secondary: `conversion` (0.29%).

**Features.** `f0..f11` only (anonymized floats; scaling for LR, raw for HistGB).

**Treatment usage.** NONE. `treatment`, `exposure`, and outcome columns never enter any feature matrix.

**Method.**
- (a) LogisticRegression (StandardScaler, max_iter=500) — fit on a 2M-row deterministic subsample (seed 42) of treated TRAIN rows; documented choice for tractability.
- (b) HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early stopping on) — fit on the FULL treated TRAIN split (~10.1M rows), no subsampling.
- Both models additionally fit for `conversion`.

**Split usage.** Frozen deterministic split by `row_hash % 1000`: train <700, validation 700–849, test >=850. ALL metrics below are on VALIDATION; the test split is never scored (row counts reported only).

**Metrics.** ROC-AUC, log loss, precision@10%, lift@10% (precision@10 / validation base rate), Brier score, decile lift table, 10-bin calibration curve, permutation importance (descriptive).

**Limitations.** Scores are predictive associations — no causal reading possible without randomization-based contrast. LR uses a subsample (slight calibration noise possible). Anonymized features: no interpretation of individual feature meanings. Response ranking implicitly weights sure-things equal to persuadables; see E3.

In [1]:
import time
T0 = time.time()
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, log_loss, brier_score_loss,
                             roc_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
import joblib

PARQUET = r"C:\Users\lucas\Desktop\ds project\Criteo\criteo_uplift_v2_hashed.parquet"
FIGDIR = r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\figures"
TABDIR = r"C:\Users\lucas\Desktop\ds project\Criteo\outputs\tables"
INTERIM = r"C:\Users\lucas\Desktop\ds project\Criteo\data\interim"
FEATURES = [f"f{i}" for i in range(12)]
SEED = 42


In [2]:
df = pd.read_parquet(PARQUET)
print(df.shape)
df["split_kind"] = np.where(df["row_hash"] % 1000 < 700, "train",
                    np.where(df["row_hash"] % 1000 < 850, "validation", "test"))
print(df["split_kind"].value_counts())
print("test rows (NOT touched below, counts only):", (df["split_kind"] == "test").sum())

train_treated = df[(df["split_kind"] == "train") & (df["treatment"] == 1)]
valid = df[df["split_kind"] == "validation"]
print("train treated rows:", len(train_treated), " validation rows (all):", len(valid))
print("validation visit base rate:", round(valid["visit"].mean(), 5))
print("validation conversion base rate:", round(valid["conversion"].mean(), 5))

# freeze working arrays, then free the big frame
Xtr_full = train_treated[FEATURES].to_numpy(dtype=np.float32)
ytr_visit = train_treated["visit"].to_numpy()
ytr_conv = train_treated["conversion"].to_numpy()
Xval = valid[FEATURES].to_numpy(dtype=np.float32)
yval_visit = valid["visit"].to_numpy()
yval_conv = valid["conversion"].to_numpy()
del df, train_treated, valid
import gc; gc.collect()
print("arrays frozen")

(13979592, 17)


split_kind
train         9789387
validation    2096296
test          2093909
Name: count, dtype: int64
test rows (NOT touched below, counts only): 2093909


train treated rows: 8322746  validation rows (all): 2096296
validation visit base rate: 0.0471
validation conversion base rate: 0.00289


arrays frozen


## 1. Model fitting — `visit`

LR: deterministic 2M-row subsample (seed 42) of the ~10.1M treated TRAIN rows — documented tractability choice. HistGB: full treated TRAIN split, no subsampling.

In [3]:
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Xtr_full), size=2_000_000, replace=False)
Xtr_lr = Xtr_full[idx]
y_visit_lr = ytr_visit[idx]
y_conv_lr = ytr_conv[idx]
print("LR subsample:", Xtr_lr.shape)

LR subsample: (2000000, 12)


In [4]:
lr_visit = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
])
t0 = time.time()
lr_visit.fit(Xtr_lr, y_visit_lr)
print("LR visit fit minutes:", round((time.time() - t0) / 60, 2))

LR visit fit minutes: 0.04


In [5]:
hgb_visit = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, early_stopping=True, random_state=SEED)
t0 = time.time()
hgb_visit.fit(Xtr_full, ytr_visit)
print("HistGB visit fit minutes:", round((time.time() - t0) / 60, 2))
print("n_iter_ used (early stopping active):", hgb_visit.n_iter_)

HistGB visit fit minutes: 2.95
n_iter_ used (early stopping active): 300


## 2. Validation metrics — `visit` models

All metrics on the VALIDATION split (treated + control rows; all rows enter denominators). Control rows never touch fitting — they appear here only because validation is the selection/evaluation set.

In [6]:
def topk_precision(y, p, k=0.10):
    n = int(len(p) * k)
    top = np.argsort(-p)[:n]
    return y[top].mean()

def evaluate(name, outcome, y_val, p_val, rows):
    prec10 = topk_precision(y_val, p_val)
    base = y_val.mean()
    return {"model": name, "outcome": outcome, "rows": rows,
            "auc": roc_auc_score(y_val, p_val),
            "log_loss": log_loss(y_val, p_val),
            "precision@10": prec10,
            "lift@10": prec10 / base,
            "brier": brier_score_loss(y_val, p_val)}

metrics = []
p_val_lr_visit = lr_visit.predict_proba(Xval)[:, 1]
metrics.append(evaluate("LogisticRegression", "visit", yval_visit, p_val_lr_visit, len(yval_visit)))
p_val_hgb_visit = hgb_visit.predict_proba(Xval)[:, 1]
metrics.append(evaluate("HistGradientBoosting", "visit", yval_visit, p_val_hgb_visit, len(yval_visit)))
pd.DataFrame(metrics).round(5)

,model,outcome,rows,auc,log_loss,precision@10,lift@10,brier
0,LogisticRegression,visit,2096296,0.93101,0.11197,0.35542,7.54572,0.03148
1,HistGradientBoosting,visit,2096296,0.94665,0.10310,0.36783,7.80904,0.02980


## 3. Same recipe for `conversion` (secondary outcome)

In [7]:
lr_conv = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
])
t0 = time.time()
lr_conv.fit(Xtr_lr, y_conv_lr)
print("LR conversion fit minutes:", round((time.time() - t0) / 60, 2))

LR conversion fit minutes: 0.05


In [8]:
hgb_conv = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=0.05, early_stopping=True, random_state=SEED)
t0 = time.time()
hgb_conv.fit(Xtr_full, ytr_conv)
print("HistGB conversion fit minutes:", round((time.time() - t0) / 60, 2))
print("n_iter_ used (early stopping active):", hgb_conv.n_iter_)

HistGB conversion fit minutes: 1.29
n_iter_ used (early stopping active): 120


In [9]:
p_val_lr_conv = lr_conv.predict_proba(Xval)[:, 1]
metrics.append(evaluate("LogisticRegression", "conversion", yval_conv, p_val_lr_conv, len(yval_conv)))
p_val_hgb_conv = hgb_conv.predict_proba(Xval)[:, 1]
metrics.append(evaluate("HistGradientBoosting", "conversion", yval_conv, p_val_hgb_conv, len(yval_conv)))
metrics_df = pd.DataFrame(metrics)
metrics_df.round(5)

,model,outcome,rows,auc,log_loss,precision@10,lift@10,brier
0,LogisticRegression,visit,2096296,0.93101,0.11197,0.35542,7.54572,0.03148
1,HistGradientBoosting,visit,2096296,0.94665,0.10310,0.36783,7.80904,0.02980
2,LogisticRegression,conversion,2096296,0.95179,0.01240,0.02549,8.83600,0.00257
3,HistGradientBoosting,conversion,2096296,0.96064,0.01159,0.02579,8.93852,0.00249


## 4. Decile lift table — best `visit` model (HistGB)

In [10]:
def decile_lift_table(y, p):
    # deciles of predicted score -> observed outcome rate per decile
    q = np.quantile(p, np.linspace(0, 1, 11))
    q[0], q[-1] = -np.inf, np.inf  # protect against float-identical boundary scores
    dec = pd.cut(p, bins=q, include_lowest=True, right=False, labels=False) + 1
    base = y.mean()
    rows = []
    for d in range(10, 0, -1):
        m = dec == d
        rows.append({"decile": d, "n": int(m.sum()),
                     "observed_rate": y[m].mean(),
                     "rel_to_base": y[m].mean() / base})
    return pd.DataFrame(rows)

lift_df = decile_lift_table(yval_visit, p_val_hgb_visit)
lift_df.round(5)

,decile,n,observed_rate,rel_to_base
0,10,209630,0.36782,7.80900
1,9,209630,0.06779,1.43912
2,8,209842,0.01855,0.39376
3,7,210013,0.00736,0.15618
4,6,209404,0.00436,0.09246
5,5,209698,0.00229,0.04860
6,4,209265,0.00115,0.02445
7,3,212966,0.00061,0.01296
8,2,208223,0.00056,0.01193
9,1,207625,0.00052,0.01094


## 5. Figures — ROC + calibration; decile lift

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

# Panel A: ROC curves, LR vs HistGB (visit, validation)
for p, name, ls, c in [(p_val_lr_visit, "LogisticRegression", "--", "C1"),
                       (p_val_hgb_visit, "HistGradientBoosting", "-", "C0")]:
    fpr, tpr, _ = roc_curve(yval_visit, p)
    axes[0].plot(fpr, tpr, ls, color=c,
                 label=f"{name} (AUC {roc_auc_score(yval_visit, p):.4f})")
axes[0].plot([0, 1], [0, 1], ":", color="grey", lw=1, label="chance")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC on validation — visit")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

# Panel B: calibration curve, HistGB visit (10 quantile bins)
frac_true, mean_pred = calibration_curve(yval_visit, p_val_hgb_visit,
                                         n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac_true, "o-", label="HistGradientBoosting")
axes[1].plot([0, 0.5], [0, 0.5], ":", color="grey", lw=1, label="perfect")
axes[1].set_xlabel("Mean predicted visit probability")
axes[1].set_ylabel("Observed visit rate")
axes[1].set_title("Calibration on validation — visit (HistGB)")
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle("Response models predict who visits — not who the ad persuades", y=1.02)
fig.tight_layout()
fig.savefig(FIGDIR + r"\e2_response_roc_calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved e2_response_roc_calibration.png")

saved e2_response_roc_calibration.png


C:\Users\lucas\AppData\Local\Temp\ipykernel_9240\1970422275.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))
base = yval_visit.mean()
x = lift_df["decile"].to_numpy()
rates = lift_df["observed_rate"].to_numpy()
top_lift = lift_df.loc[lift_df["decile"] == 10, "rel_to_base"].iloc[0]
colors = ["C3"] * 3 + ["C0"] * 7  # top 3 deciles highlighted
ax.bar(x, rates, color=colors)
ax.axhline(base, color="grey", ls="--", lw=1.2,
           label=f"validation base rate ({base:.3%})")
for xi, r in zip(x, rates):
    ax.annotate(f"{r:.1%}", (xi, r), ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"D{d}\n{rel:.2f}x" for d, rel in
                    zip(lift_df["decile"], lift_df["rel_to_base"])])
ax.set_xlabel("Predicted-score decile (D10 = top 10%) — relative-to-base annotated")
ax.set_ylabel("Observed visit rate")
ax.set_title(f"Top-decile users are ~{top_lift:.1f}x as likely to visit; response is concentrated in D9–D10, not spread across mid deciles")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGDIR + r"\e2_response_decile_lift.png", dpi=150, bbox_inches="tight")
plt.show()
print("saved e2_response_decile_lift.png")

saved e2_response_decile_lift.png


C:\Users\lucas\AppData\Local\Temp\ipykernel_9240\2582034775.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Descriptive feature importance (HistGB, permutation, 500k validation rows)

Descriptive only — f0..f11 are anonymized; we record which positions carry signal and invent **no** meaning. Not part of any decision beyond "signal exists"; definitely not causal.

In [13]:
samp = np.random.default_rng(SEED).choice(len(Xval), size=500_000, replace=False)
t0 = time.time()
perm = permutation_importance(hgb_visit, Xval[samp], yval_visit[samp],
                              scoring="neg_log_loss", n_repeats=5,
                              random_state=SEED, n_jobs=1)
print("permutation importance minutes:", round((time.time() - t0) / 60, 2))
imp_df = (pd.DataFrame({"feature": FEATURES,
                        "importance_mean": perm["importances_mean"],
                        "importance_std": perm["importances_std"]})
          .sort_values("importance_mean", ascending=False)
          .reset_index(drop=True))
imp_df.round(6)

permutation importance minutes: 3.01


,feature,importance_mean,importance_std
0,f8,0.074384,0.000179
1,f2,0.028993,0.000106
2,f9,0.007062,0.000037
3,f6,0.001680,0.000070
4,f0,0.001503,0.000049
5,f3,0.000244,0.000004
6,f11,0.000063,0.000004
7,f4,0.000063,0.000004
8,f10,0.000048,0.000010
9,f7,0.000041,0.000004


## 7. Save artifacts (tables + best visit model)

In [14]:
metrics_df.to_csv(TABDIR + r"\e2_response_metrics.csv", index=False)
lift_df.to_csv(TABDIR + r"\e2_response_lift_deciles.csv", index=False)
imp_df.to_csv(TABDIR + r"\e2_feature_importance.csv", index=False)
print("tables saved to outputs/tables/")

tables saved to outputs/tables/


In [15]:
best_visit = {
    "model": hgb_visit,
    "model_class": "HistGradientBoostingClassifier",
    "hyperparams": {"max_iter": 300, "learning_rate": 0.05,
                    "early_stopping": True, "random_state": SEED},
    "outcome": "visit",
    "fit_population": "train-split treated rows only (row_hash % 1000 < 700, treatment == 1)",
    "features": FEATURES,
    "validation_metrics": {
        r["model"]: {k: float(v) for k, v in r.items()
                     if k not in ("model", "outcome", "rows")}
        for _, r in metrics_df[metrics_df["outcome"] == "visit"].iterrows()
    },
}
joblib.dump(best_visit, INTERIM + r"\e2_best_visit_model.joblib")
print("saved e2_best_visit_model.joblib")
print("total notebook minutes:", round((time.time() - T0) / 60, 2))

saved e2_best_visit_model.joblib
total notebook minutes: 7.97


## Does predicting conversion tell us who to target?

**No — not by itself.** P(Y=1 | X) mixes two very different kinds of users: those who would visit/buy *anyway* (**sure things** — the ad spends budget on response it did not cause) and those moved *by* the ad (**persuadables** — the only users the campaign can win). A response score ranks the two together, and its top decile can even be dominated by sure things.

The relevant targeting quantity is a *difference of potential outcomes* — P(Y=1|X, treat) vs P(Y=1|X, control) — not a single conditional prediction. **E3** takes exactly that step: estimating uplift, separating sure things from persuadables, and re-deciding the top-k rule on that basis.